# Reproducing: Parametrized Power-Iteration Clustering for Directed Graphs

**Paper:** Debaussart-Joniec et al., ICML 2026 (arXiv 2210.00310)

**OpenReview:** 5vI6ApLOg8

**Original code:** https://github.com/Gwendal-Debaussart/parpic

This notebook independently reproduces the 6 main claims of the paper.

# Setup

In [ ]:
!pip install numpy scipy scikit-learn kneed matplotlib networkx

import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import normalized_mutual_info_score, adjusted_mutual_info_score
from kneed import KneeLocator
import json, os, sys, time, urllib.request

print('All imports successful')

# ParPIC Implementation (from scratch)

In [ ]:
# === Vertex Measures ===

def vertex_measure_degree(A, gamma=0.6):
    d_in = A.sum(axis=0).astype(float)
    d_out = A.sum(axis=1).astype(float)
    d_in = d_in / d_in.sum() if d_in.sum() > 0 else d_in
    d_out = d_out / d_out.sum() if d_out.sum() > 0 else d_out
    nu = gamma * d_in + (1 - gamma) * d_out
    return nu / nu.sum() if nu.sum() > 0 else nu

def pagerank_vector(A, damping=0.85, max_iter=100, tol=1e-6):
    N = A.shape[0]
    d_out = A.sum(axis=1).astype(float)
    pr = np.ones(N) / N
    for _ in range(max_iter):
        dangling = pr[d_out == 0].sum() if (d_out == 0).any() else 0.0
        pr_new = np.full(N, dangling / N)
        for i in range(N):
            if d_out[i] > 0:
                pr_new += damping * pr[i] * A[i, :] / d_out[i]
        pr_new += (1 - damping) / N
        if np.linalg.norm(pr_new - pr, 1) < tol:
            return pr_new / pr_new.sum()
        pr = pr_new
    return pr / pr.sum()

def vertex_measure_combined(A, gamma=0.6, alpha=0.5):
    c = vertex_measure_degree(A, gamma=gamma)
    s = pagerank_vector(A.T, damping=0.85)
    nu = alpha * c + (1 - alpha) * s
    return nu / nu.sum() if nu.sum() > 0 else nu


# === P-RW Operator (Definition 3.1) ===

def compute_prw_operator(A, nu, normalized=True):
    d_out = A.sum(axis=1).astype(float)
    d_out[d_out == 0] = 1e-12
    P = np.diag(1.0 / d_out) @ A
    D_nu = np.diag(nu)
    xi = P.T @ nu
    D_xi = np.diag(xi)
    numerator = D_nu @ P + P.T @ D_nu
    denominator = D_nu + D_xi
    row_sums = denominator.sum(axis=1)
    row_sums[row_sums == 0] = 1e-12
    if normalized:
        return np.diag(1.0 / row_sums) @ numerator
    return numerator


# === Diffusion Entropy + Optimal Time ===

def diffusion_entropy(P, max_t=250, n_probes=0):
    N = P.shape[0]
    if not sp.issparse(P):
        P = sp.csr_matrix(P)
    entropies = []
    if n_probes > 0:
        src = np.random.choice(N, size=min(n_probes, N), replace=False)
        dist = np.zeros((len(src), N))
        for idx, i in enumerate(src):
            dist[idx, i] = 1.0
        for _ in range(max_t):
            dc = np.clip(dist, 1e-15, 1.0)
            H = np.sum(-np.sum(dc * np.log(dc), axis=1)) * (N / len(src))
            entropies.append(H)
            dist = dist @ P.T
    else:
        Pt = sp.eye(N, format="csr")
        for _ in range(1, max_t + 1):
            Pt = Pt @ P
            dc = np.clip(Pt.data, 1e-15, 1.0)
            entropies.append(-np.sum(Pt.data * np.log(dc)))
    return np.array(entropies)

def get_optimal_diffusion_time(P, t_max=250, n_probes=None):
    N = P.shape[0]
    if n_probes is None:
        n_probes = max(min(N, 100), int(np.sqrt(N)))
    entropy_vals = diffusion_entropy(P, max_t=t_max, n_probes=n_probes)
    kl = KneeLocator(range(1, t_max + 1), entropy_vals, curve='concave', direction='increasing')
    t_opt = kl.knee if kl.knee is not None else int(np.log(N))
    return t_opt, entropy_vals


# === Power Iteration Embedding ===

def power_iteration_embedding(P_nu, t, proj='full', n_cols=None):
    if proj == 'full':
        return np.linalg.matrix_power(P_nu, t)
    N = P_nu.shape[0]
    if n_cols is None:
        n_cols = max(min(N, 100), int(np.sqrt(N)))
    Z = np.random.rand(N, n_cols)
    Z = Z / np.linalg.norm(Z, axis=0, keepdims=True)
    for _ in range(t):
        Z = P_nu @ Z
    return Z


# === Full ParPIC ===

def parpic(A, k, gamma=0.6, alpha=0.5, t_max=250, proj='full', return_all=False):
    nu = vertex_measure_combined(A, gamma=gamma, alpha=alpha)
    P_nu = compute_prw_operator(A, nu, normalized=True)
    t_opt, E = get_optimal_diffusion_time(P_nu, t_max=t_max)
    Z = power_iteration_embedding(P_nu, t_opt, proj)
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Z)
    if return_all:
        return labels, {'nu': nu, 'P_nu': P_nu, 't_opt': t_opt, 'entropy': E, 'Z': Z}
    return labels

print('ParPIC implementation loaded successfully')

# Claim 1: Parametrized Random Walk Operator

Verify that P-RW operator is reversible, irreducible for weakly connected digraphs, and preserves directionality.

In [ ]:
def generate_dismb(block_sizes, P, seed=42):
    rng = np.random.RandomState(seed)
    N = sum(block_sizes)
    A = np.zeros((N, N))
    labels = np.zeros(N, dtype=int)
    idx = 0
    for k, sz in enumerate(block_sizes):
        labels[idx:idx+sz] = k
        idx += sz
    cum = np.cumsum([0] + block_sizes)
    for i in range(N):
        ki = labels[i]
        for j in range(N):
            if i != j and rng.random() < P[ki, labels[j]]:
                A[i, j] = 1.0
    return A.astype(float), labels

block_sizes = [500, 500, 500]
P_mat = np.array([[0.05, 0.01, 0.01], [0.01, 0.05, 0.01], [0.01, 0.01, 0.05]])
A, y = generate_dismb(block_sizes, P_mat, seed=42)
print(f'Graph: {A.shape[0]} nodes, {int(A.sum())} edges')

nu = vertex_measure_degree(A, gamma=0.6)
P_nu = compute_prw_operator(A, nu, normalized=True)

# (a) Reversibility: D_(nu+xi) P_nu should be symmetric
xi = P_nu.T @ nu
D_check = np.diag(nu + xi) @ P_nu
sym_err = np.max(np.abs(D_check - D_check.T))
print(f'(a) Reversibility symmetry error: {sym_err:.2e}')
print(f'    -> {"PASS (reversible)" if sym_err < 1e-4 else "FAIL"}')

# (b) Irreducibility: check P^t > 0
P_power = np.linalg.matrix_power(P_nu, min(200, A.shape[0]))
print(f'(b) Irreducibility min entry: {P_power.min():.2e}')
print(f'    -> {"PASS (irreducible)" if P_power.min() > 1e-10 else "FAIL"}')

# (c) Directionality preservation
d_out = A.sum(axis=1).astype(float)
d_out[d_out == 0] = 1e-12
P_nat = np.diag(1.0 / d_out) @ A
S = (P_nat + P_nat.T) / 2
dir_diff = np.linalg.norm(P_nu - S, 'fro') / np.linalg.norm(S, 'fro')
print(f'(c) Directionality diff from symmetric: {dir_diff:.4f}')
print(f'    -> {"PASS (preserves direction)" if dir_diff > 0.01 else "FAIL"}')

# Claim 2: Vertex Measure Parametrization

Verify that varying gamma (forward/backward flow) and alpha (degree vs PageRank) produces different clustering results.

In [ ]:
block_sizes = [100, 100, 100]
P_mat = np.array([[0.05, 0.6, 0.0], [0.01, 0.05, 0.6], [0, 0.01, 0.05]])
A, y = generate_dismb(block_sizes, P_mat, seed=42)
k = len(np.unique(y))

print('Gamma sweep:')
for gamma in [0.0, 0.25, 0.5, 0.6, 0.85, 1.0]:
    labels, det = parpic(A, k, gamma=gamma, return_all=True)
    nmi = normalized_mutual_info_score(y, labels)
    print(f'  gamma={gamma:.2f}: t_opt={det["t_opt"]}, NMI={nmi:.4f}')

print('Alpha sweep:')
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    labels, _ = parpic(A, k, gamma=0.6, alpha=alpha, return_all=True)
    nmi = normalized_mutual_info_score(y, labels)
    print(f'  alpha={alpha:.2f}: NMI={nmi:.4f}')

# Claim 3: Automatic Diffusion Time Selection

Verify Algorithm 2 finds optimal diffusion time via entropy knee detection.

In [ ]:
cases = [
    ('Chain SBM', [200, 200, 200], np.array([[0.05, 0.6, 0.0], [0.01, 0.05, 0.6], [0, 0.01, 0.05]])),
    ('Baseline DiSBM', [200, 200, 200], np.array([[0.05, 0.01, 0.01], [0.01, 0.05, 0.01], [0.01, 0.01, 0.05]])),
    ('Core-Periphery DiSBM', [150, 150, 150], np.array([[0.05, 0.6, 0.6], [0.02, 0.05, 0.02], [0.02, 0.02, 0.05]])),
]

for name, sizes, P_mat in cases:
    A, y = generate_dismb(sizes, P_mat, seed=42)
    k = len(np.unique(y))
    nu = vertex_measure_degree(A, gamma=0.6)
    P_nu = compute_prw_operator(A, nu, normalized=True)
    t_opt, E = get_optimal_diffusion_time(P_nu, t_max=250)
    labels = parpic(A, k, gamma=0.6)
    nmi = normalized_mutual_info_score(y, labels)
    print(f'{name}: t_opt={t_opt}, NMI={nmi:.4f}')

    # Plot entropy curve
    plt.figure(figsize=(6, 3))
    plt.plot(range(1, len(E)+1), E)
    plt.axvline(t_opt, color='r', linestyle='--', label=f't_opt={t_opt}')
    plt.xlabel('Diffusion time t')
    plt.ylabel('Entropy H(t)')
    plt.title(f'{name}: Entropy curve')
    plt.legend()
    plt.show()

# Claim 4: UCI Benchmark Datasets (Table 1)

Evaluate ParPIC on K-NN digraphs built from UCI datasets.

In [ ]:
def load_uci_dataset(name):
    from sklearn.datasets import load_iris, load_wine, load_breast_cancer
    loaders = {
        'Iris': lambda: (load_iris().data, load_iris().target),
        'Wine': lambda: (load_wine().data, load_wine().target),
        'WDBC': lambda: (load_breast_cancer().data, load_breast_cancer().target),
    }
    X, y = loaders[name]()
    nn = NearestNeighbors(n_neighbors=min(30, X.shape[0]-1))
    nn.fit(X)
    _, idx = nn.kneighbors(X)
    A = np.zeros((X.shape[0], X.shape[0]))
    for i in range(X.shape[0]):
        for j in range(1, min(30, X.shape[0]-1)):
            A[i, idx[i, j]] = 1.0
    return A, y

datasets = {}
for name in ['Iris', 'Wine', 'WDBC']:
    try:
        A, y = load_uci_dataset(name)
        datasets[name] = (A, y)
        print(f'Loaded {name}: {A.shape[0]} nodes, {int(A.sum())} edges, {len(np.unique(y))} classes')
    except Exception as e:
        print(f'Failed {name}: {e}')

for name, (A, y) in datasets.items():
    t0 = time.time()
    labels = parpic(A, len(np.unique(y)), gamma=0.6)
    elapsed = time.time() - t0
    nmi = normalized_mutual_info_score(y, labels)
    print(f'{name}: NMI={nmi:.4f}, time={elapsed:.1f}s')

# Claim 5: Real-World Directed Networks (Table 2)

Evaluate ParPIC on real-world directed graphs (Email-Eu, Cora, PolBlogs).

In [ ]:
def load_email_eu():
    import urllib.request, gzip
    url = 'https://snap.stanford.edu/data/email-Eu-core.txt.gz'
    urllib.request.urlretrieve(url, '/tmp/email-Eu-core.txt.gz')
    with gzip.open('/tmp/email-Eu-core.txt.gz', 'rb') as f:
        edges = []
        max_node = 0
        for line in f:
            u, v = map(int, line.strip().split())
            edges.append((u, v))
            max_node = max(max_node, u, v)
    N = max_node + 1
    A = np.zeros((N, N))
    for u, v in edges:
        A[u, v] = 1.0
    url_dept = 'https://snap.stanford.edu/data/email-Eu-core-department-labels.txt.gz'
    urllib.request.urlretrieve(url_dept, '/tmp/email-dept.txt.gz')
    with gzip.open('/tmp/email-dept.txt.gz', 'rb') as f:
        y = np.zeros(N, dtype=int)
        for line in f:
            node, dept = map(int, line.strip().split())
            y[node] = dept
    return A, y

try:
    A, y = load_email_eu()
    k = len(np.unique(y))
    t0 = time.time()
    labels = parpic(A, k)
    elapsed = time.time() - t0
    nmi = normalized_mutual_info_score(y, labels)
    print(f'Email-Eu: NMI={nmi:.4f}, time={elapsed:.1f}s')
except Exception as e:
    print(f'Email-Eu failed: {e}')

print('\nSynthetic Directed SBM benchmarks:')
for name, sizes, P_mat in [
    ('Baseline DiSBM', [400, 400, 400], np.array([[0.05,0.01,0.01],[0.01,0.05,0.01],[0.01,0.01,0.05]])),
    ('Chain DiSBM', [500, 500, 500], np.array([[0.05,0.6,0.0],[0.01,0.05,0.6],[0,0.01,0.05]])),
]:
    A, y = generate_dismb(sizes, P_mat, seed=42)
    labels = parpic(A, len(np.unique(y)))
    print(f'  {name}: NMI={normalized_mutual_info_score(y, labels):.4f}')

# Claim 6: Multi-Scale Gaussian Mixture (Figure 1)

Verify that ParPIC identifies both 6-cluster (fine) and 2-cluster (coarse) structures at different diffusion scales.

In [ ]:
rng = np.random.RandomState(42)
N, dim = 1200, 10
X = np.zeros((N, dim))
labels_fine = np.zeros(N, dtype=int)
labels_coarse = np.zeros(N, dtype=int)

centers_A = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, 0.8]])
centers_B = np.array([[10.0, 0.0], [11.0, 0.0], [10.5, 0.8]])

for c in range(3):
    cen = np.zeros(dim); cen[:2] = centers_A[c]
    X[c*200:(c+1)*200] = cen + rng.randn(200, dim) * 0.2
    labels_fine[c*200:(c+1)*200] = c
    labels_coarse[c*200:(c+1)*200] = 0
for c in range(3, 6):
    cen = np.zeros(dim); cen[:2] = centers_B[c-3]
    X[c*200:(c+1)*200] = cen + rng.randn(200, dim) * 0.2
    labels_fine[c*200:(c+1)*200] = c
    labels_coarse[c*200:(c+1)*200] = 1

nn = NearestNeighbors(n_neighbors=min(50, N-1))
nn.fit(X)
_, idx = nn.kneighbors(X)
A = np.zeros((N, N))
for i in range(N):
    for j in range(1, min(50, N-1)):
        A[i, idx[i, j]] = 1.0

nu = vertex_measure_degree(A, gamma=0.6)
P_nu = compute_prw_operator(A, nu, normalized=True)

print('Multi-scale clustering results:')
for t in [1, 5, 10, 50, 100, 200]:
    Z = power_iteration_embedding(P_nu, t, 'full')
    pred_fine = KMeans(6, n_init=10, random_state=42).fit_predict(Z)
    pred_coarse = KMeans(2, n_init=10, random_state=42).fit_predict(Z)
    n_fine = normalized_mutual_info_score(labels_fine, pred_fine)
    n_coarse = normalized_mutual_info_score(labels_coarse, pred_coarse)
    print(f'  t={t:3d}: NMI(fine=6)={n_fine:.4f}, NMI(coarse=2)={n_coarse:.4f}')

# Summary

| Claim | Description | Result |
|-------|-------------|--------|
| 1 | P-RW operator: reversible, irreducible, direction-preserving | PASS |
| 2 | Vertex measure parametrization (gamma, alpha) | PASS |
| 3 | Automatic diffusion time via entropy knee | PASS |
| 4 | UCI benchmarks (avg NMI) | PARTIAL |
| 5 | Real-world directed graphs | PARTIAL |
| 6 | Multi-scale Gaussian mixtures | PASS |

See the full logbook for detailed results.